# Waste Management AI — 3-Class Object Detection Prototype

**Purpose:** Build a small, reproducible object-detection pipeline that can be trained in Google Colab and moved to an HPC computer.

### Prototype classes
- `plastic` → represented initially by **bottle**
- `fabric` → represented initially by **clothing/fabric-like objects**
- `organic` → represented initially by **watermelon**

> **Important:** A bottle is not automatically plastic, and clothing is not identical to fabric. This is only a prototype mapping. For the competition model, replace these proxy examples with images of the actual waste materials and objects from your game field.

### Pipeline
`data → labels → train/val/test split → pretrained YOLO → training → validation → error analysis → export → HPC`


## 0. Recommended prototype targets

These are **engineering targets**, not guaranteed results.

- **Minimum acceptable validation mAP50:** 0.80
- **Preferred validation mAP50:** ≥ 0.90
- **Target inference speed:** ≥ 20 FPS on the final robot computer
- **Camera:** 30 FPS
- **Deployment:** onboard robot computer; the exact board/GPU should be confirmed later
- **Failure cases to test:** poor lighting, shadows, object partly hidden, objects touching/overlapping, small/far objects, unusual orientations, motion blur, dirty/crushed objects, confusing background, and false detections on field boards/floor.

For the competition, do not choose the final hardware or FPS target until the actual robot computer and camera pipeline are benchmarked.


## 1. Install the software

This installs Ultralytics for YOLO training and FiftyOne for downloading a small, filtered subset of Open Images.

We deliberately **do not download the full Open Images dataset**. It is enormous. We only request small subsets containing the classes we need for this prototype.


In [ ]:
!pip -q install -U ultralytics fiftyone


## 2. Imports and project configuration

This cell creates the folders used by the notebook and defines the three classes.

Class IDs must remain consistent:
- `0 = plastic`
- `1 = fabric`
- `2 = organic`


In [ ]:
from pathlib import Path
import random
import shutil
import yaml
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

PROJECT = Path("/content/waste_ai")
RAW_DIR = PROJECT / "raw"
DATASET_DIR = PROJECT / "dataset"
EXPORT_DIR = PROJECT / "exports"

for p in [RAW_DIR, DATASET_DIR, EXPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

CLASSES = ["plastic", "fabric", "organic"]
CLASS_TO_ID = {name: i for i, name in enumerate(CLASSES)}

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Project:", PROJECT)
print("Classes:", CLASS_TO_ID)


Project: /content/waste_ai
Classes: {'plastic': 0, 'fabric': 1, 'organic': 2}


## 3. Download a small public dataset subset

We use **Open Images V7** through FiftyOne.

For this prototype:
- `Bottle` → `plastic`
- `Watermelon` → `organic`
- `Clothing` → temporary `fabric` proxy

The download is intentionally small. You can increase the sample counts later.

Open Images provides bounding-box annotations, so this is suitable for object detection rather than just image classification.


In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz

def load_open_images_subset(class_name, max_samples=80, seed=42):
    name = "open-images-v7"
    print(f"Loading {max_samples} samples for {class_name}...")
    return foz.load_zoo_dataset(
        name,
        split="validation",
        label_types=["detections"],
        classes=[class_name],
        only_matching=True,
        max_samples=max_samples,
        shuffle=True,
        seed=seed,
    )

bottle_fo = load_open_images_subset("Bottle", max_samples=80, seed=42)
watermelon_fo = load_open_images_subset("Watermelon", max_samples=80, seed=43)
clothing_fo = load_open_images_subset("Clothing", max_samples=80, seed=44)

print("Bottle samples:", len(bottle_fo))
print("Watermelon samples:", len(watermelon_fo))
print("Clothing samples:", len(clothing_fo))


/usr/local/lib/python3.13/dist-packages/glob2/fnmatch.py:141: SyntaxWarning: invalid escape sequence '\Z'
  return '(?ms)' + res + '\Z'


ImportError: cannot import name '_Ink' from 'PIL._typing' (/usr/local/lib/python3.13/dist-packages/PIL/_typing.py)

## 4. Inspect the public data before training

Do not train blindly.

Look at a few examples and ask:
- Does the object look like what the robot camera will see?
- Are the boxes correct?
- Is the object too easy to detect because of a clean background?
- Are there weird examples that should be removed?

This inspection step becomes more important when you move to your real competition dataset.


In [ ]:
# FiftyOne can open an interactive browser-style viewer in some Colab setups.
# If the viewer does not work in your runtime, continue with the saved dataset
# and inspect exported images later.

print("Open Images subsets loaded successfully.")
print("Next, export the relevant bounding boxes into our 3-class YOLO dataset.")


## 5. Create YOLO train/validation/test folders

Ultralytics YOLO expects images and matching label files.

Each label line is:

`class_id x_center y_center width height`

with all coordinates normalized from 0 to 1.


In [ ]:
for split in ["train", "val", "test"]:
    (DATASET_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (DATASET_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

print("YOLO dataset folders created.")


## 6. Export FiftyOne detections into our three-class dataset

The function below:
1. reads each image,
2. keeps only the requested Open Images class,
3. converts the bounding box to YOLO format,
4. assigns our competition-prototype class ID,
5. copies the image into the dataset.

We use a deterministic split so the same source image is not randomly moved around every run.


In [ ]:
def export_fiftyone_class(dataset, source_class, target_class, max_images=None):
    target_id = CLASS_TO_ID[target_class]
    samples = list(dataset)
    if max_images:
        samples = samples[:max_images]

    exported = []
    for sample in samples:
        src = Path(sample.filepath)
        if not src.exists():
            continue

        # Make a unique filename so datasets from different sources do not collide.
        out_name = f"{target_class}_{src.stem}_{src.suffix.lower().replace('.', '')}{src.suffix.lower()}"

        # Save image temporarily; split happens later.
        tmp_dir = RAW_DIR / target_class
        tmp_dir.mkdir(parents=True, exist_ok=True)
        out_img = tmp_dir / out_name
        if not out_img.exists():
            shutil.copy2(src, out_img)

        detections = []
        # Open Images is loaded into FiftyOne with detection labels.
        if hasattr(sample, "ground_truth") and sample.ground_truth is not None:
            for det in sample.ground_truth.detections:
                if det.label.lower() == source_class.lower():
                    x, y, w, h = det.bounding_box
                    xc = x + w / 2
                    yc = y + h / 2
                    detections.append(
                        f"{target_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}"
                    )

        # Some FiftyOne Open Images imports use another field name.
        if not detections:
            for field_name in sample.field_names:
                value = sample[field_name]
                if hasattr(value, "detections"):
                    for det in value.detections:
                        if det.label.lower() == source_class.lower():
                            x, y, w, h = det.bounding_box
                            detections.append(
                                f"{target_id} {x+w/2:.6f} {y+h/2:.6f} {w:.6f} {h:.6f}"
                            )
                    if detections:
                        break

        if detections:
            label_path = out_img.with_suffix(".txt")
            label_path.write_text("\n".join(detections) + "\n")
            exported.append(out_img)

    return exported

# Export the three prototype sources.
plastic_files = export_fiftyone_class(bottle_fo, "Bottle", "plastic")
organic_files = export_fiftyone_class(watermelon_fo, "Watermelon", "organic")
fabric_files = export_fiftyone_class(clothing_fo, "Clothing", "fabric")

print("Exported:")
print("plastic:", len(plastic_files))
print("fabric:", len(fabric_files))
print("organic:", len(organic_files))


## 7. IMPORTANT: add your own fabric images

The public `Clothing` class is only a temporary proxy.

For the next iteration, upload photographs of the **actual fabric material** you want the robot to recognize.

If you already have a YOLO-labeled fabric dataset, place it in:

`/content/waste_ai/raw/fabric_real/`

with:
- images
- matching `.txt` labels

If you only have unlabelled fabric photographs, annotate them first with a tool such as CVAT or Roboflow.

For this first smoke test, we continue with the Clothing proxy.


## 8. Split the dataset

We create:
- 70% training
- 20% validation
- 10% test

The test set is kept separate and should only be used for the final evaluation.

For the real competition dataset, split by **scene/video/session**, not just random adjacent frames, to avoid leaking nearly identical images into train and test.


In [ ]:
def collect_labeled_images():
    items = []
    for class_name in CLASSES:
        class_dir = RAW_DIR / class_name
        if not class_dir.exists():
            continue
        for img in class_dir.iterdir():
            if img.suffix.lower() not in [".jpg", ".jpeg", ".png", ".webp"]:
                continue
            label = img.with_suffix(".txt")
            if label.exists() and label.read_text().strip():
                items.append((img, label))
    return items

items = collect_labeled_images()
random.Random(SEED).shuffle(items)

n = len(items)
n_train = int(0.70 * n)
n_val = int(0.20 * n)

splits = {
    "train": items[:n_train],
    "val": items[n_train:n_train+n_val],
    "test": items[n_train+n_val:]
}

for split, pairs in splits.items():
    for img_path, label_path in pairs:
        dst_img = DATASET_DIR / "images" / split / img_path.name
        dst_lbl = DATASET_DIR / "labels" / split / label_path.name
        shutil.copy2(img_path, dst_img)
        shutil.copy2(label_path, dst_lbl)

    print(split, len(pairs))

print("Total:", n)


## 9. Check class balance

A model can look good overall while performing badly on one class.

We count labels for each class before training.


In [ ]:
from collections import Counter

counts = Counter()

for split in ["train", "val", "test"]:
    for label_file in (DATASET_DIR / "labels" / split).glob("*.txt"):
        for line in label_file.read_text().splitlines():
            if line.strip():
                class_id = int(line.split()[0])
                counts[CLASSES[class_id]] += 1

print("Object counts:")
for name in CLASSES:
    print(f"{name:10s}: {counts[name]}")


## 10. Create `data.yaml`

This file tells YOLO:
- where the training images are,
- where validation images are,
- where test images are,
- what the class names are.

Keep this file with the dataset when you move the project to HPC.


In [ ]:
data_yaml = {
    "path": str(DATASET_DIR),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {
        0: "plastic",
        1: "fabric",
        2: "organic",
    }
}

yaml_path = DATASET_DIR / "data.yaml"
yaml_path.write_text(yaml.safe_dump(data_yaml, sort_keys=False))

print(yaml_path.read_text())


## 11. Visual sanity check

Before training, display a few training images with their bounding boxes.

If the boxes or class IDs are wrong, **stop here and fix the dataset**.
Training a bad dataset faster does not produce a better model.


In [ ]:
def show_samples(split="train", n=6):
    image_dir = DATASET_DIR / "images" / split
    images = list(image_dir.glob("*"))
    random.Random(SEED).shuffle(images)
    images = images[:n]

    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    axes = axes.flatten()

    for ax, img_path in zip(axes, images):
        img = Image.open(img_path).convert("RGB")
        ax.imshow(img)
        ax.set_title(img_path.name)
        ax.axis("off")

        label_path = DATASET_DIR / "labels" / split / f"{img_path.stem}.txt"
        if label_path.exists():
            W, H = img.size
            for line in label_path.read_text().splitlines():
                cls, xc, yc, w, h = map(float, line.split()[:5])
                x1 = (xc - w/2) * W
                y1 = (yc - h/2) * H
                bw = w * W
                bh = h * H
                rect = plt.Rectangle((x1, y1), bw, bh, fill=False, linewidth=2)
                ax.add_patch(rect)
                ax.text(x1, y1, CLASSES[int(cls)], fontsize=9,
                        bbox=dict(facecolor="white", alpha=0.7))

    plt.tight_layout()
    plt.show()

show_samples()


## 12. Train a small pretrained YOLO model

We start with a **small pretrained detection model** so the prototype can train quickly.

For a smoke test:
- `epochs=30` is enough to confirm the pipeline works.
- Later, the HPC run can use more epochs and a larger model if needed.

Do not judge final competition performance from this tiny prototype.


In [ ]:
from ultralytics import YOLO

MODEL_NAME = "yolo26n.pt"

model = YOLO(MODEL_NAME)

results = model.train(
    data=str(yaml_path),
    epochs=30,
    imgsz=640,
    batch=-1,
    patience=10,
    project=str(PROJECT / "runs"),
    name="prototype",
    exist_ok=True,
)

print("Training finished.")


## 13. Validate the trained model

Validation answers:

> "How well does the model detect the classes on images it did not train on?"

Pay attention to:
- mAP50
- mAP50-95
- precision
- recall
- per-class performance

Do not look only at one overall number.


In [ ]:
best_model_path = PROJECT / "runs" / "prototype" / "weights" / "best.pt"
best_model = YOLO(str(best_model_path))

metrics = best_model.val(
    data=str(yaml_path),
    split="val",
    imgsz=640,
)

print("mAP50-95:", metrics.box.map)
print("mAP50:", metrics.box.map50)
print("Per-class mAP50-95:", metrics.box.maps)


## 14. Test on completely unseen images

The test set should be treated as the final exam.

Do this after model development, not repeatedly while tuning the model.


In [ ]:
test_metrics = best_model.val(
    data=str(yaml_path),
    split="test",
    imgsz=640,
)

print("TEST mAP50-95:", test_metrics.box.map)
print("TEST mAP50:", test_metrics.box.map50)
print("TEST per-class mAP50-95:", test_metrics.box.maps)


## 15. Run inference on test images

Now look at what the robot would actually see.

The important question is not just "what is the metric?"

It is:

> **Does the model put the correct box around the correct waste object?**


In [ ]:
test_images = list((DATASET_DIR / "images" / "test").glob("*"))

predictions = best_model.predict(
    source=[str(p) for p in test_images[:12]],
    imgsz=640,
    conf=0.25,
    save=True,
    project=str(PROJECT / "predictions"),
    name="test_predictions",
    exist_ok=True,
)

print("Prediction images saved to:", PROJECT / "predictions" / "test_predictions")


## 16. Measure inference speed

Your camera is 30 FPS, but that does **not** mean the AI must necessarily run at 30 FPS.

For a robotics system, we first want a stable real-time perception loop. Benchmark the model on the **actual deployment computer** later.

This Colab timing is only a rough development benchmark.


In [ ]:
import time

if test_images:
    sample = str(test_images[0])

    start = time.perf_counter()
    for _ in range(10):
        _ = best_model.predict(source=sample, imgsz=640, verbose=False)
    elapsed = time.perf_counter() - start

    fps = 10 / elapsed
    print(f"Approximate Colab inference throughput: {fps:.2f} FPS")
    print("This is NOT the final robot FPS.")


## 17. Save the model for the HPC

The important artifact is:

`best.pt`

Also save:
- `data.yaml`
- the dataset
- training results
- this notebook
- a requirements file
- a README describing the experiment

This makes the experiment reproducible on another computer.


In [ ]:
requirements = 'ultralytics\nfiftyone\npyyaml\nnumpy\npillow\nmatplotlib\n'
(EXPORT_DIR / "requirements.txt").write_text(requirements)

# Copy the best model into a simple export location.
hpc_model = EXPORT_DIR / "best.pt"
shutil.copy2(best_model_path, hpc_model)

# Copy data.yaml
shutil.copy2(yaml_path, EXPORT_DIR / "data.yaml")

print("HPC model:", hpc_model)
print("Requirements:", EXPORT_DIR / "requirements.txt")


## 18. Package the project

The ZIP produced here can be transferred to the HPC system.

For a large real dataset, do not repeatedly copy the same dataset unnecessarily. Store the dataset in the HPC's designated storage area and keep the code/configuration under version control.


In [ ]:
import zipfile

zip_path = EXPORT_DIR / "waste_ai_prototype_for_hpc.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    # Dataset
    for p in DATASET_DIR.rglob("*"):
        if p.is_file():
            z.write(p, p.relative_to(PROJECT))

    # Best model
    z.write(hpc_model, hpc_model.relative_to(PROJECT))

    # Requirements
    req = EXPORT_DIR / "requirements.txt"
    z.write(req, req.relative_to(PROJECT))

print("Package created:", zip_path)


# What to improve next

### Version 1 — this notebook
Small proof of concept:
- 3 classes
- public proxy data
- pretrained YOLO
- Colab training
- validation
- test
- model export

### Version 2 — real objects
Replace the proxy classes with:
- actual plastic bottle photographs
- actual fabric waste/material photographs
- actual organic waste photographs

### Version 3 — competition environment
Collect images using the **actual robot camera** on the actual game field:
- field boards
- floor
- lighting
- shadows
- distance
- object placement
- screen/target information
- clutter

### Version 4 — robotics integration
Connect:

`camera → YOLO → object position → ROS2 → navigation/manipulation`

### Version 5 — HPC training
Move the same code/configuration to HPC:
- create the Python environment
- transfer/attach dataset
- use the HPC GPU
- run `train.py` as a batch job
- save checkpoints and `best.pt`

### Version 6 — deployment
Benchmark on the robot computer and decide whether you need:
- a smaller model,
- lower image size,
- ONNX,
- TensorRT,
- or another optimized deployment format.

### Version 7 — robustness
Add hard examples and repeat:
`collect → label → train → validate → inspect errors → collect more → retrain`.


# Experiment record

For every training run, record:

- Dataset version:
- Number of images:
- Images per class:
- Model:
- Epochs:
- Image size:
- Batch size:
- GPU:
- mAP50:
- mAP50-95:
- Precision:
- Recall:
- Per-class performance:
- Inference FPS:
- Main failure cases:
- What changed from the previous run:

This record is what will allow your team to compare HPC experiments instead of guessing which model was best.
